In [75]:
import torch

In [76]:
data = torch.load('data_tensor.pt')

In [78]:
data.shape[0]

1749370

In [95]:
data[:2,]

tensor([[[0., 0., 0., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 1., 2., 0., 0., 0., 0.],
         [1., 0., 1., 0., 2., 0., 0., 0., 0., 0., 0., 0., 0., 2., 1., 0., 0.,
          0., 0., 0., 0., 0., 0., 0.]],

        [[1., 0., 0., 0., 0., 1., 0., 0., 0., 2., 0., 0., 1., 0., 0., 0., 0.,
          0., 0., 0., 1., 0., 0., 0.],
         [0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 4., 1.]]])

In [113]:
def return_accuracy(w):
    if type(w) is not torch.Tensor:
        w = torch.tensor(w, dtype=torch.float32)
    y_true = torch.zeros(data.shape[0], dtype=torch.long)
    U = torch.matmul(data, w)                
    y_pred = torch.argmax(U, dim=1)      
    correct = (U[:, 0] - U[:, 1]) > 0
    accuracy = correct.float().mean()
    return accuracy.item()

In [119]:
import numpy as np

def interpret_vector(weight_vector, normalize=True):
    """
    Interprets a weight vector for the trolley problem utility function.
    
    Args:
        weight_vector (list/array): Weights corresponding to features
        normalize (bool): Whether to normalize people weights to show relative preferences
        
    Returns:
        str: Human-readable interpretation of the moral preferences encoded in the weights
    """
    
    # Feature names for reference
    feature_names = ['Intervention', 'PedPed', 'Barrier', 'CrossingSignal', 'Man', 'Woman', 
                    'Pregnant', 'Stroller', 'OldMan', 'OldWoman', 'Boy', 'Girl', 'Homeless', 
                    'LargeWoman', 'LargeMan', 'Criminal', 'MaleExecutive', 'FemaleExecutive', 
                    'FemaleAthlete', 'MaleAthlete', 'FemaleDoctor', 'MaleDoctor', 'Dog', 'Cat']
    
    if len(weight_vector) != len(feature_names):
        raise ValueError(f"Weight vector length ({len(weight_vector)}) doesn't match feature count ({len(feature_names)})")
    
    weights = np.array(weight_vector)
    
    # Separate structural and people weights
    structural_weights = weights[:4]
    people_weights = weights[4:]
    people_names = feature_names[4:]
    
    # Normalize people weights if requested
    if normalize and len(people_weights) > 0:
        people_sum = np.sum(np.abs(people_weights))
        if people_sum > 0:
            normalized_people = people_weights / people_sum
        else:
            normalized_people = people_weights
    else:
        normalized_people = people_weights
    
    output_lines = []
    output_lines.append("=== TROLLEY PROBLEM MORAL PREFERENCES ===\n")
    
    # Analyze structural preferences
    interv_weight = structural_weights[0]
    pedped_weight = structural_weights[1] 
    barrier_weight = structural_weights[2]
    crossing_weight = structural_weights[3]
    
    structural_insights = []
    
    if abs(interv_weight) > 0.05:
        if interv_weight > 0:
            structural_insights.append(f"Favors taking action over inaction (weight: {interv_weight:.2f})")
        else:
            structural_insights.append(f"Prefers inaction over intervention (weight: {interv_weight:.2f})")
    
    if abs(barrier_weight) > 0.05:
        if barrier_weight > 0:
            structural_insights.append(f"Values passengers over pedestrians (weight: {barrier_weight:.2f})")
        else:
            structural_insights.append(f"Values pedestrians over passengers (weight: {barrier_weight:.2f})")
    
    if abs(crossing_weight) > 0.05:
        if crossing_weight > 0:
            structural_insights.append(f"Strongly considers legal vs illegal crossing (weight: {crossing_weight:.2f})")
        else:
            structural_insights.append(f"Penalizes legal behavior - counterintuitive (weight: {crossing_weight:.2f})")
    
    if abs(pedped_weight) > 0.05:
        if pedped_weight > 0:
            structural_insights.append(f"Prefers pedestrian-vs-pedestrian scenarios (weight: {pedped_weight:.2f})")
        else:
            structural_insights.append(f"Prefers pedestrian-vs-passenger scenarios (weight: {pedped_weight:.2f})")
    
    if structural_insights:
        output_lines.append("STRUCTURAL PREFERENCES:")
        for insight in structural_insights:
            output_lines.append(f"  • {insight}")
        output_lines.append("")
    
    # Analyze people preferences
    if len(normalized_people) > 0:
        # Sort by preference strength
        sorted_indices = np.argsort(normalized_people)[::-1]  # Descending order
        
        # Group into categories
        highly_valued = []
        moderately_valued = []
        devalued = []
        neutral = []
        
        for idx in sorted_indices:
            name = people_names[idx]
            weight = normalized_people[idx]
            
            if weight > 0.08:  # Highly valued
                highly_valued.append((name, weight))
            elif weight > 0.03:  # Moderately valued
                moderately_valued.append((name, weight))
            elif weight < -0.03:  # Devalued
                devalued.append((name, weight))
            else:  # Neutral
                neutral.append((name, weight))
        
        output_lines.append("PERSON TYPE PREFERENCES:")
        
        if highly_valued:
            output_lines.append("  HIGHLY VALUED:")
            for name, weight in highly_valued:
                output_lines.append(f"    • {name}: {weight:.3f}")
        
        if moderately_valued:
            output_lines.append("  MODERATELY VALUED:")
            for name, weight in moderately_valued:
                output_lines.append(f"    • {name}: {weight:.3f}")
        
        if devalued:
            output_lines.append("  DEVALUED:")
            for name, weight in devalued:
                output_lines.append(f"    • {name}: {weight:.3f}")
        
        output_lines.append("")
    
    # Generate key insights
    insights = []
    
    # Check for demographic patterns
    children_weights = [normalized_people[people_names.index(name)] for name in ['Boy', 'Girl', 'Stroller'] if name in people_names]
    adult_weights = [normalized_people[people_names.index(name)] for name in ['Man', 'Woman'] if name in people_names]
    elderly_weights = [normalized_people[people_names.index(name)] for name in ['OldMan', 'OldWoman'] if name in people_names]
    
    if children_weights and adult_weights:
        avg_children = np.mean(children_weights)
        avg_adults = np.mean(adult_weights)
        if avg_children > avg_adults + 0.02:
            insights.append(f"Shows strong preference for children over adults (children: {avg_children:.3f}, adults: {avg_adults:.3f})")
        elif avg_adults > avg_children + 0.02:
            insights.append(f"Prioritizes adults over children (adults: {avg_adults:.3f}, children: {avg_children:.3f})")
    
    # Check professional bias
    professionals = ['FemaleDoctor', 'MaleDoctor', 'FemaleExecutive', 'MaleExecutive']
    prof_weights = [normalized_people[people_names.index(name)] for name in professionals if name in people_names]
    if prof_weights:
        avg_prof = np.mean(prof_weights)
        if avg_prof > 0.06:
            insights.append(f"Values professionals highly (avg: {avg_prof:.3f})")
    
    # Check for gender bias
    male_types = ['Man', 'OldMan', 'Boy', 'LargeMan', 'MaleExecutive', 'MaleAthlete', 'MaleDoctor']
    female_types = ['Woman', 'OldWoman', 'Girl', 'LargeWoman', 'FemaleExecutive', 'FemaleAthlete', 'FemaleDoctor']
    
    male_weights = [normalized_people[people_names.index(name)] for name in male_types if name in people_names]
    female_weights = [normalized_people[people_names.index(name)] for name in female_types if name in people_names]
    
    if male_weights and female_weights:
        avg_male = np.mean(male_weights)
        avg_female = np.mean(female_weights)
        if abs(avg_male - avg_female) > 0.02:
            if avg_male > avg_female:
                insights.append(f"Shows slight male bias (male: {avg_male:.3f}, female: {avg_female:.3f})")
            else:
                insights.append(f"Shows slight female bias (female: {avg_female:.3f}, male: {avg_male:.3f})")
    
    # Check for vulnerable populations
    vulnerable = ['Homeless', 'Criminal']
    vuln_weights = [normalized_people[people_names.index(name)] for name in vulnerable if name in people_names]
    if vuln_weights:
        avg_vuln = np.mean(vuln_weights)
        if avg_vuln < -0.01:
            insights.append(f"Devalues marginalized groups (avg: {avg_vuln:.3f})")
    
    # Check animal consideration
    animals = ['Dog', 'Cat']
    animal_weights = [normalized_people[people_names.index(name)] for name in animals if name in people_names]
    if animal_weights:
        avg_animal = np.mean(animal_weights)
        if avg_animal > 0.02:
            insights.append(f"Assigns significant value to animal lives (avg: {avg_animal:.3f})")
    
    if insights:
        output_lines.append("KEY MORAL INSIGHTS:")
        for insight in insights:
            output_lines.append(f"  • {insight}")
        output_lines.append("")
    
    # Overall moral philosophy summary
    output_lines.append("MORAL PHILOSOPHY SUMMARY:")
    
    # Determine overall approach
    if interv_weight > 0.1:
        output_lines.append("  • Utilitarian approach - willing to actively intervene to maximize outcomes")
    elif interv_weight < -0.1:
        output_lines.append("  • Deontological approach - prefers not to actively cause harm")
    else:
        output_lines.append("  • Mixed approach to intervention vs. inaction")
    
    if crossing_weight > 0.1:
        output_lines.append("  • Rule-based ethics - values legal and social compliance")
    
    # Find the most extreme preference
    if len(normalized_people) > 0:
        max_idx = np.argmax(np.abs(normalized_people))
        max_weight = normalized_people[max_idx]
        max_name = people_names[max_idx]
        
        if max_weight > 0:
            output_lines.append(f"  • Strongest positive bias: {max_name} (weight: {max_weight:.3f})")
        else:
            output_lines.append(f"  • Strongest negative bias: {max_name} (weight: {max_weight:.3f})")
    
    output_lines.append(f"\nUtility Calculation: Higher scores = more likely to save that option")
    output_lines.append("Decision Rule: Choose the option (save vs. alternative) with higher utility score")
    
    return "\n".join(output_lines)

# Example usage
def demo_interpretation():
    """Demonstrates the interpreter with example weights"""
    
    example_weights = [
        0.3,   # Intervention (prefer action)
        0.0,   # PedPed (neutral)
        -0.2,  # Barrier (prefer pedestrians)  
        0.4,   # CrossingSignal (value legal behavior)
        1.0,   # Man (baseline)
        1.0,   # Woman (equal)
        2.2,   # Pregnant (highly valued)
        2.5,   # Stroller (children most valued)
        0.8,   # OldMan 
        0.8,   # OldWoman   
        2.0,   # Boy (children valued)
        2.0,   # Girl (children valued)
        0.4,   # Homeless (devalued)
        1.0,   # LargeWoman 
        1.0,   # LargeMan
        0.1,   # Criminal (strongly devalued)
        1.3,   # MaleExecutive (professionals valued)
        1.3,   # FemaleExecutive 
        1.0,   # FemaleAthlete
        1.0,   # MaleAthlete
        1.8,   # FemaleDoctor (professionals highly valued)
        1.8,   # MaleDoctor
        0.3,   # Dog 
        0.3    # Cat 
    ]
    
    result = interpret_vector(example_weights)
    print(result)
    return result

# Run demo if script is executed directly
if __name__ == "__main__":
    demo_interpretation()

=== TROLLEY PROBLEM MORAL PREFERENCES ===

STRUCTURAL PREFERENCES:
  • Favors taking action over inaction (weight: 0.30)
  • Values pedestrians over passengers (weight: -0.20)
  • Strongly considers legal vs illegal crossing (weight: 0.40)

PERSON TYPE PREFERENCES:
  HIGHLY VALUED:
    • Stroller: 0.106
    • Pregnant: 0.093
    • Girl: 0.085
    • Boy: 0.085
  MODERATELY VALUED:
    • MaleDoctor: 0.076
    • FemaleDoctor: 0.076
    • FemaleExecutive: 0.055
    • MaleExecutive: 0.055
    • LargeWoman: 0.042
    • Woman: 0.042
    • Man: 0.042
    • LargeMan: 0.042
    • FemaleAthlete: 0.042
    • MaleAthlete: 0.042
    • OldWoman: 0.034
    • OldMan: 0.034

KEY MORAL INSIGHTS:
  • Shows strong preference for children over adults (children: 0.092, adults: 0.042)
  • Values professionals highly (avg: 0.066)

MORAL PHILOSOPHY SUMMARY:
  • Utilitarian approach - willing to actively intervene to maximize outcomes
  • Rule-based ethics - values legal and social compliance
  • Strongest posit

In [132]:
# Openai

vector = [-0.3, 0.1, -0.5, 0.4, 1.0, 1.0, 1.6, 1.8, 0.7, 0.7, 1.5, 1.5, 0.5, 0.8, 0.8, -0.6, 1.2, 1.2, 1.4, 1.4, 1.3, 1.3, 0.6, 0.6]
print(return_accuracy(vector))
print(interpret_vector(vector))

0.7434196472167969
=== TROLLEY PROBLEM MORAL PREFERENCES ===

STRUCTURAL PREFERENCES:
  • Prefers inaction over intervention (weight: -0.30)
  • Values pedestrians over passengers (weight: -0.50)
  • Strongly considers legal vs illegal crossing (weight: 0.40)
  • Prefers pedestrian-vs-pedestrian scenarios (weight: 0.10)

PERSON TYPE PREFERENCES:
  HIGHLY VALUED:
    • Stroller: 0.084
  MODERATELY VALUED:
    • Pregnant: 0.074
    • Girl: 0.070
    • Boy: 0.070
    • MaleAthlete: 0.065
    • FemaleAthlete: 0.065
    • MaleDoctor: 0.060
    • FemaleDoctor: 0.060
    • FemaleExecutive: 0.056
    • MaleExecutive: 0.056
    • Man: 0.047
    • Woman: 0.047
    • LargeWoman: 0.037
    • LargeMan: 0.037
    • OldWoman: 0.033
    • OldMan: 0.033

KEY MORAL INSIGHTS:
  • Shows strong preference for children over adults (children: 0.074, adults: 0.047)
  • Assigns significant value to animal lives (avg: 0.028)

MORAL PHILOSOPHY SUMMARY:
  • Deontological approach - prefers not to actively cause h

# Method 1 : Random Sampling

In [114]:
import numpy as np
import time

def random_search_optimizer(n_dimensions=24, n_iterations=10000, bounds=(-10.0, 10.0)):
    """
    Performs a random search to find the best weight vector.

    Args:
        n_dimensions (int): The number of dimensions in the weight vector.
        n_iterations (int): The number of random vectors to test.
        bounds (tuple): A tuple (min_val, max_val) for the random weights.

    Returns:
        tuple: A tuple containing (best_weights, best_accuracy).
    """
    print("--- Starting Random Search ---")
    start_time = time.time()
    
    best_accuracy = -1.0
    best_weights = None

    for i in range(n_iterations):
        # Generate a random weight vector within the specified bounds
        weights = np.random.uniform(bounds[0], bounds[1], n_dimensions)
        
        # Get the accuracy for this vector
        accuracy = return_accuracy(weights)
        
        # If it's the best we've seen, save it
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_weights = weights
            print(f"Iteration {i+1}/{n_iterations}: New best accuracy = {accuracy:.6f}")

    end_time = time.time()
    print(f"--- Random Search Finished in {end_time - start_time:.2f} seconds ---")
    return best_weights, best_accuracy

In [125]:
random_weights, accuracy = random_search_optimizer()

--- Starting Random Search ---
Iteration 1/10000: New best accuracy = 0.457102
Iteration 2/10000: New best accuracy = 0.559771
Iteration 3/10000: New best accuracy = 0.595469
Iteration 6/10000: New best accuracy = 0.602013
Iteration 14/10000: New best accuracy = 0.624220
Iteration 77/10000: New best accuracy = 0.662342
Iteration 319/10000: New best accuracy = 0.675397
Iteration 2026/10000: New best accuracy = 0.688939
Iteration 4071/10000: New best accuracy = 0.691563
Iteration 4750/10000: New best accuracy = 0.692026
Iteration 7063/10000: New best accuracy = 0.708216
Iteration 9486/10000: New best accuracy = 0.714313
--- Random Search Finished in 157.71 seconds ---


In [126]:
s = interpret_vector(random_weights)
print(s)

=== TROLLEY PROBLEM MORAL PREFERENCES ===

STRUCTURAL PREFERENCES:
  • Prefers inaction over intervention (weight: -4.54)
  • Values passengers over pedestrians (weight: 0.14)
  • Penalizes legal behavior - counterintuitive (weight: -2.75)
  • Prefers pedestrian-vs-pedestrian scenarios (weight: 6.58)

PERSON TYPE PREFERENCES:
  HIGHLY VALUED:
    • FemaleAthlete: 0.113
    • MaleAthlete: 0.111
    • FemaleExecutive: 0.106
    • Pregnant: 0.094
    • Homeless: 0.085
    • FemaleDoctor: 0.082
  MODERATELY VALUED:
    • LargeMan: 0.059
    • OldMan: 0.056
    • Girl: 0.050
    • Stroller: 0.050
    • LargeWoman: 0.050
    • Man: 0.036
    • MaleExecutive: 0.032

MORAL PHILOSOPHY SUMMARY:
  • Deontological approach - prefers not to actively cause harm
  • Strongest positive bias: FemaleAthlete (weight: 0.113)

Utility Calculation: Higher scores = more likely to save that option
Decision Rule: Choose the option (save vs. alternative) with higher utility score


# Method 2 - Hill Climbing

In [128]:
import numpy as np
import time

# (Mock return_accuracy function would be here)

def hill_climbing_optimizer(n_dimensions=24, n_iterations=10000, bounds=(-10.0, 10.0), step_size=0.01, initial_weights=None):
    """
    Performs a hill climbing search.

    Args:
        n_dimensions (int): The number of dimensions.
        n_iterations (int): The number of iterations to try and find a better neighbor.
        bounds (tuple): A tuple (min_val, max_val) for the weights.
        step_size (float): The magnitude of the random change at each step.

    Returns:
        tuple: A tuple containing (best_weights, best_accuracy).
    """
    print("--- Starting Hill Climbing ---")
    start_time = time.time()
    
    # 1. Start with a random vector
    if initial_weights is not None:
        current_weights = np.array(initial_weights)
    else:
        current_weights = np.random.uniform(bounds[0], bounds[1], n_dimensions)
    current_accuracy = return_accuracy(current_weights)
    print(f"Initial accuracy: {current_accuracy:.6f}")

    # 2. Loop and try to find better neighbors
    for i in range(n_iterations):
        # Create a new vector by making a small, random change (a "step")
        noise = np.random.normal(0, step_size, n_dimensions)
        new_weights = current_weights + noise
        
        # Clip the values to stay within the defined bounds
        new_weights = np.clip(new_weights, bounds[0], bounds[1])

        # Evaluate the new vector
        new_accuracy = return_accuracy(new_weights)
        
        # If the new vector is better, move to that position
        if new_accuracy > current_accuracy:
            current_weights = new_weights
            current_accuracy = new_accuracy
            print(f"Iteration {i+1}/{n_iterations}: New best accuracy = {current_accuracy:.6f}")

    end_time = time.time()
    print(f"--- Hill Climbing Finished in {end_time - start_time:.2f} seconds ---")
    return current_weights, current_accuracy

In [129]:
hill_weights, accuracy = hill_climbing_optimizer(initial_weights=random_weights)
print(f"Best accuracy found: {accuracy:.6f}")

--- Starting Hill Climbing ---
Initial accuracy: 0.714313
Iteration 3/10000: New best accuracy = 0.714519
Iteration 10/10000: New best accuracy = 0.714587
Iteration 11/10000: New best accuracy = 0.714625
Iteration 14/10000: New best accuracy = 0.714687
Iteration 15/10000: New best accuracy = 0.714739
Iteration 16/10000: New best accuracy = 0.714746
Iteration 17/10000: New best accuracy = 0.714864
Iteration 18/10000: New best accuracy = 0.714924
Iteration 20/10000: New best accuracy = 0.715081
Iteration 22/10000: New best accuracy = 0.715103
Iteration 23/10000: New best accuracy = 0.715267
Iteration 27/10000: New best accuracy = 0.715291
Iteration 28/10000: New best accuracy = 0.715616
Iteration 29/10000: New best accuracy = 0.715658
Iteration 30/10000: New best accuracy = 0.715740
Iteration 31/10000: New best accuracy = 0.715754
Iteration 32/10000: New best accuracy = 0.715767
Iteration 36/10000: New best accuracy = 0.715850
Iteration 37/10000: New best accuracy = 0.715869
Iteration 41

In [130]:
# trying another iteration
hill_weights, accuracy = hill_climbing_optimizer(initial_weights=hill_weights)
print(f"Best accuracy found: {accuracy:.6f}")

--- Starting Hill Climbing ---
Initial accuracy: 0.780285
Iteration 1/10000: New best accuracy = 0.780303
Iteration 20/10000: New best accuracy = 0.780312
Iteration 32/10000: New best accuracy = 0.780343
Iteration 53/10000: New best accuracy = 0.780343
Iteration 85/10000: New best accuracy = 0.780359
Iteration 97/10000: New best accuracy = 0.780359
Iteration 145/10000: New best accuracy = 0.780374
Iteration 149/10000: New best accuracy = 0.780378
Iteration 195/10000: New best accuracy = 0.780380
Iteration 230/10000: New best accuracy = 0.780387
Iteration 262/10000: New best accuracy = 0.780389
Iteration 266/10000: New best accuracy = 0.780399
Iteration 271/10000: New best accuracy = 0.780402
Iteration 380/10000: New best accuracy = 0.780411
Iteration 386/10000: New best accuracy = 0.780418
Iteration 390/10000: New best accuracy = 0.780421
Iteration 393/10000: New best accuracy = 0.780439
Iteration 402/10000: New best accuracy = 0.780441
Iteration 586/10000: New best accuracy = 0.780444

In [131]:
s = interpret_vector(hill_weights)
print(s)

=== TROLLEY PROBLEM MORAL PREFERENCES ===

STRUCTURAL PREFERENCES:
  • Favors taking action over inaction (weight: 0.31)
  • Values pedestrians over passengers (weight: -3.86)
  • Penalizes legal behavior - counterintuitive (weight: -2.94)
  • Prefers pedestrian-vs-pedestrian scenarios (weight: 6.09)

PERSON TYPE PREFERENCES:
  HIGHLY VALUED:
    • Pregnant: 0.093
    • Stroller: 0.080
  MODERATELY VALUED:
    • Girl: 0.070
    • FemaleAthlete: 0.067
    • MaleAthlete: 0.066
    • FemaleDoctor: 0.064
    • Boy: 0.063
    • FemaleExecutive: 0.061
    • MaleDoctor: 0.058
    • Woman: 0.049
    • LargeWoman: 0.048
    • MaleExecutive: 0.048
    • Man: 0.047
    • LargeMan: 0.045
    • Homeless: 0.045
    • OldMan: 0.038

KEY MORAL INSIGHTS:
  • Shows strong preference for children over adults (children: 0.071, adults: 0.048)

MORAL PHILOSOPHY SUMMARY:
  • Utilitarian approach - willing to actively intervene to maximize outcomes
  • Strongest positive bias: Pregnant (weight: 0.093)

Utilit

# Method 3: Genetic Algorithm

In [ ]:
import random
import time
import numpy as np
from deap import base, creator, tools, algorithms

# (Mock return_accuracy function would be here)

# --- DEAP Setup ---
# We are trying to MAXIMIZE accuracy, so weights are 1.0
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

# Attribute generator: each weight is a float between -10 and 10
N_DIMENSIONS = 24
BOUND_LOW, BOUND_UP = -10.0, 10.0
toolbox.register("attr_float", random.uniform, BOUND_LOW, BOUND_UP)

# Structure initializers
# An "Individual" is a list of N_DIMENSIONS floats
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_float, n=N_DIMENSIONS)
# A "population" is a list of individuals
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# --- Genetic Operators ---
# Evaluation function
def eval_accuracy(individual):
    # DEAP works with lists, but our function expects a numpy array
    weights = np.array(individual)
    return (return_accuracy(weights),) # Must return a tuple

toolbox.register("evaluate", eval_accuracy)
# Crossover operator
toolbox.register("mate", tools.cxTwoPoint)
# Mutation operator
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=0.1, indpb=0.1)
# Selection operator
toolbox.register("select", tools.selTournament, tournsize=3)

def genetic_algorithm_optimizer(pop_size=200, n_generations=200):
    """
    Uses a Genetic Algorithm to find the best weight vector.
    """
    print("--- Starting Genetic Algorithm ---")
    start_time = time.time()

    pop = toolbox.population(n=pop_size)
    # Track the best individual found
    hof = tools.HallOfFame(1)
    # Gather statistics
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("std", np.std)
    stats.register("min", np.min)
    stats.register("max", np.max)

    # Run the evolutionary algorithm
    algorithms.eaSimple(pop, toolbox, cxpb=0.5, mutpb=0.2, ngen=n_generations, 
                        stats=stats, halloffame=hof, verbose=True)

    end_time = time.time()
    print(f"--- Genetic Algorithm Finished in {end_time - start_time:.2f} seconds ---")
    
    best_individual = hof[0]
    best_accuracy = best_individual.fitness.values[0]
    best_weights = np.array(best_individual)
    
    return best_weights, best_accuracy

/opt/miniconda3/envs/dharma/lib/python3.13/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/opt/miniconda3/envs/dharma/lib/python3.13/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


In [136]:
genetic_weights, accuracy = genetic_algorithm_optimizer()

--- Starting Genetic Algorithm ---
gen	nevals	avg     	std      	min     	max     
0  	200   	0.498906	0.0609439	0.360077	0.653766
1  	114   	0.548616	0.0478289	0.432541	0.68579 
2  	115   	0.593681	0.0406133	0.478258	0.68579 
3  	120   	0.627672	0.0336076	0.554937	0.697939
4  	122   	0.655735	0.0239999	0.548394	0.697939
5  	101   	0.674832	0.0181248	0.57817 	0.717198
6  	108   	0.689582	0.0117061	0.650005	0.721055
7  	127   	0.698074	0.0111997	0.653611	0.726304
8  	134   	0.705975	0.0113066	0.659491	0.733973
9  	106   	0.714263	0.00955151	0.685706	0.743172
10 	109   	0.72089 	0.00821451	0.69568 	0.749573
11 	126   	0.726635	0.00955466	0.697092	0.752975
12 	109   	0.733416	0.00815297	0.702802	0.755386
13 	111   	0.739864	0.00722922	0.709905	0.755898
14 	123   	0.745241	0.00596095	0.727961	0.758763
15 	112   	0.750268	0.00451895	0.735376	0.762324
16 	118   	0.753739	0.00304543	0.74202 	0.762207
17 	128   	0.756036	0.0026689 	0.744535	0.76295 
18 	114   	0.758129	0.00237256	0.752751	0.76

In [137]:
print(interpret_vector(genetic_weights))

=== TROLLEY PROBLEM MORAL PREFERENCES ===

STRUCTURAL PREFERENCES:
  • Favors taking action over inaction (weight: 0.14)
  • Values pedestrians over passengers (weight: -5.66)
  • Penalizes legal behavior - counterintuitive (weight: -4.19)
  • Prefers pedestrian-vs-pedestrian scenarios (weight: 4.12)

PERSON TYPE PREFERENCES:
  HIGHLY VALUED:
    • Stroller: 0.083
  MODERATELY VALUED:
    • Pregnant: 0.075
    • FemaleDoctor: 0.070
    • MaleDoctor: 0.066
    • FemaleExecutive: 0.061
    • Boy: 0.058
    • Girl: 0.058
    • FemaleAthlete: 0.056
    • MaleExecutive: 0.054
    • Woman: 0.051
    • Man: 0.051
    • LargeWoman: 0.049
    • MaleAthlete: 0.049
    • Homeless: 0.049
    • LargeMan: 0.048
    • OldMan: 0.046
    • OldWoman: 0.031

KEY MORAL INSIGHTS:
  • Values professionals highly (avg: 0.063)

MORAL PHILOSOPHY SUMMARY:
  • Utilitarian approach - willing to actively intervene to maximize outcomes
  • Strongest positive bias: Stroller (weight: 0.083)

Utility Calculation: High

# Method 4 - Bayesian Optimization

In [140]:
import numpy as np
import time
from skopt import gp_minimize
from skopt.space import Real


def bayesian_optimizer(n_dimensions=24, n_calls=200, bounds=(-10.0, 10.0)):
    """
    Uses Bayesian Optimization to find the best weight vector.

    Args:
        n_dimensions (int): The number of dimensions.
        n_calls (int): The number of times to call return_accuracy.
        bounds (tuple): A tuple (min_val, max_val) for the weights.

    Returns:
        tuple: A tuple containing (best_weights, best_accuracy).
    """
    print("--- Starting Bayesian Optimization ---")
    start_time = time.time()
    
    # 1. Define the search space
    search_space = [Real(bounds[0], bounds[1], name=f'w_{i}') for i in range(n_dimensions)]

    # 2. Define the objective function
    # scikit-optimize performs MINIMIZATION, so we must return 1.0 - accuracy.
    def objective_function(weights):
        weights = np.array(weights)
        accuracy = return_accuracy(weights)
        return 1.0 - accuracy

    # 3. Run the optimizer
    result = gp_minimize(
        func=objective_function,
        dimensions=search_space,
        n_calls=n_calls,
        n_initial_points=10, # How many random points to probe before building the model
        random_state=42,
        verbose=True
    )

    end_time = time.time()
    print(f"--- Bayesian Optimization Finished in {end_time - start_time:.2f} seconds ---")
    
    # The result.x contains the best parameters found
    best_weights = np.array(result.x)
    # The result.fun contains the best *minimized* value (1.0 - accuracy)
    best_accuracy = 1.0 - result.fun
    
    return best_weights, best_accuracy


In [141]:
bayes_weights, accuracy = bayesian_optimizer()

--- Starting Bayesian Optimization ---
Iteration No: 1 started. Evaluating function at random point.
Iteration No: 1 ended. Evaluation done at random point.
Time taken: 0.1665
Function value obtained: 0.5979
Current minimum: 0.5979
Iteration No: 2 started. Evaluating function at random point.
Iteration No: 2 ended. Evaluation done at random point.
Time taken: 0.0211
Function value obtained: 0.5176
Current minimum: 0.5176
Iteration No: 3 started. Evaluating function at random point.
Iteration No: 3 ended. Evaluation done at random point.
Time taken: 0.0179
Function value obtained: 0.4263
Current minimum: 0.4263
Iteration No: 4 started. Evaluating function at random point.
Iteration No: 4 ended. Evaluation done at random point.
Time taken: 0.0179
Function value obtained: 0.4308
Current minimum: 0.4263
Iteration No: 5 started. Evaluating function at random point.
Iteration No: 5 ended. Evaluation done at random point.
Time taken: 0.0227
Function value obtained: 0.4816
Current minimum: 0.4

In [144]:
print(interpret_vector(bayes_weights))

=== TROLLEY PROBLEM MORAL PREFERENCES ===

STRUCTURAL PREFERENCES:
  • Favors taking action over inaction (weight: 2.99)
  • Values pedestrians over passengers (weight: -6.09)
  • Penalizes legal behavior - counterintuitive (weight: -7.09)
  • Prefers pedestrian-vs-pedestrian scenarios (weight: 10.00)

PERSON TYPE PREFERENCES:
  MODERATELY VALUED:
    • LargeWoman: 0.054
    • LargeMan: 0.054
    • Woman: 0.054
    • Pregnant: 0.054
    • Stroller: 0.054
    • OldMan: 0.054
    • Boy: 0.054
    • Girl: 0.054
    • Homeless: 0.054
    • Man: 0.054
    • Criminal: 0.054
    • MaleExecutive: 0.054
    • FemaleAthlete: 0.054
    • MaleAthlete: 0.054
    • FemaleDoctor: 0.054
    • MaleDoctor: 0.054
    • FemaleExecutive: 0.044
    • OldWoman: 0.044

KEY MORAL INSIGHTS:
  • Assigns significant value to animal lives (avg: 0.023)

MORAL PHILOSOPHY SUMMARY:
  • Utilitarian approach - willing to actively intervene to maximize outcomes
  • Strongest positive bias: Man (weight: 0.054)

Utility Ca

# Method 5 - CMA-ES

In [145]:
import numpy as np
import cma
import time


def cma_es_optimizer(n_dimensions=24, initial_guess=None, sigma=0.5, n_iterations=1000):
    """
    Uses the CMA-ES algorithm to find the best weight vector.

    Args:
        n_dimensions (int): The number of dimensions.
        initial_guess (np.ndarray): An initial starting point for the search. If None, starts at all zeros.
        sigma (float): The initial standard deviation (step size). This is a crucial parameter.
        n_iterations (int): The maximum number of iterations (function evaluations).

    Returns:
        tuple: A tuple containing (best_weights, best_accuracy).
    """
    print("--- Starting CMA-ES ---")
    start_time = time.time()

    if initial_guess is None:
        initial_guess = np.zeros(n_dimensions)

    # CMA-ES aims to MINIMIZE a function. Because our function returns
    # accuracy (higher is better), we need to minimize its negative.
    # We create a wrapper "loss" function for this purpose.
    def loss_function(weights):
        accuracy = return_accuracy(weights)
        return -accuracy # Return the negative, so minimizing it maximizes accuracy

    # Create a CMA-ES optimizer instance
    # The 'inopts' dictionary allows you to set options like bounds.
    es = cma.CMAEvolutionStrategy(initial_guess, sigma, {'bounds': [-10, 10]})

    # The optimization loop
    # We can use the 'ask-and-tell' interface for more control.
    iterations = 0
    while not es.stop() and iterations < n_iterations:
        # Ask for a new population of solutions
        solutions = es.ask()
        
        # Tell the optimizer the fitness (loss) of each solution
        # This requires calling your function for each candidate vector
        fitness_values = [loss_function(w) for w in solutions]
        es.tell(solutions, fitness_values)
        
        # Log progress and increment counter
        es.disp()
        iterations += 1

    end_time = time.time()
    print(f"--- CMA-ES Finished in {end_time - start_time:.2f} seconds ---")

    # The result is stored in the `result` property of the optimizer
    best_weights = es.result.xbest
    # The best fitness is the minimized loss (a negative number)
    best_loss = es.result.fbest
    # Convert it back to accuracy
    best_accuracy = -best_loss

    return best_weights, best_accuracy


In [146]:
cma_weights, accuracy = cma_es_optimizer()

--- Starting CMA-ES ---
(6_w,13)-aCMA-ES (mu_w=4.0,w_1=38%) in dimension 24 (seed=553889, Wed Sep 10 18:45:34 2025)
Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
    1     13 -6.037076115608215e-01 1.0e+00 4.68e-01  5e-01  5e-01 0:00.4
    2     26 -6.165087819099426e-01 1.1e+00 4.58e-01  5e-01  5e-01 0:00.7
    3     39 -6.287903785705566e-01 1.1e+00 4.54e-01  4e-01  5e-01 0:00.9
   19    247 -7.639830112457275e-01 1.5e+00 4.66e-01  4e-01  5e-01 0:03.9
   42    546 -7.798258662223816e-01 2.8e+00 7.18e-01  6e-01  9e-01 0:07.9
   71    923 -7.820815443992615e-01 4.3e+00 4.84e-01  4e-01  7e-01 0:13.0
  100   1300 -7.825525999069214e-01 4.5e+00 1.90e-01  1e-01  2e-01 0:18.0
  141   1833 -7.828955650329590e-01 5.5e+00 9.76e-02  5e-02  1e-01 0:25.0
  188   2444 -7.830613255500793e-01 6.9e+00 4.48e-02  2e-02  6e-02 0:33.2
  200   2600 -7.830567359924316e-01 7.2e+00 3.62e-02  2e-02  5e-02 0:35.3
  257   3341 -7.831001877784729e-01 8.5e+00 1.57e-02  7e-03  2e-02 0:45.

In [147]:
print(interpret_vector(cma_weights))

=== TROLLEY PROBLEM MORAL PREFERENCES ===

STRUCTURAL PREFERENCES:
  • Favors taking action over inaction (weight: 1.19)
  • Values pedestrians over passengers (weight: -5.75)
  • Penalizes legal behavior - counterintuitive (weight: -4.58)
  • Prefers pedestrian-vs-pedestrian scenarios (weight: 1.99)

PERSON TYPE PREFERENCES:
  HIGHLY VALUED:
    • Stroller: 0.086
    • Pregnant: 0.084
    • Girl: 0.083
    • Boy: 0.080
  MODERATELY VALUED:
    • FemaleDoctor: 0.066
    • MaleDoctor: 0.064
    • FemaleAthlete: 0.049
    • Woman: 0.049
    • MaleAthlete: 0.048
    • FemaleExecutive: 0.048
    • Man: 0.048
    • MaleExecutive: 0.047
    • LargeWoman: 0.047
    • LargeMan: 0.046
    • OldWoman: 0.040
    • OldMan: 0.039
    • Homeless: 0.039

KEY MORAL INSIGHTS:
  • Shows strong preference for children over adults (children: 0.083, adults: 0.048)

MORAL PHILOSOPHY SUMMARY:
  • Utilitarian approach - willing to actively intervene to maximize outcomes
  • Strongest positive bias: Stroller (